## Aligning Lyrics With Word-level Timestamps
Forced alignement with known good subtitles and an audio file.

In [ ]:
# Setup Code
import whisperx


device = "cuda" # or "cpu"

# Download the model
model = whisperx.load_model("tiny", device) # optionally, compute_type can be used, e.g. float16, int8, etc. 
# Download the alignment model and metadata
model_a, metadata = whisperx.load_align_model(language_code="en", device=device)

The code should ideally run the setup code at least once for the first time so the models can be downloaded, optionally, other models or different language alignments can be downloaded as well. 
- models: `tiny.en`, `tiny`, `base.en`, `base`, `small.en`, `small`, `medium.en`, `medium`, `large-v1`, `large-v2`, `large-v3`, `large`, `distil-large-v2`, `distil-medium.en`, `distil-small.en`, `distil-large-v3`, `distil-large-v3.5`, `large-v3-turbo`, `turbo`
- alignment models are language codes like `en`, `zh`, `ja`, etc. 

In [196]:
# All Imports
import whisperx
import json
import srt
import gc
import torch
import pylrc
from pathlib import Path
import time
from dataclasses import dataclass
from collections.abc import Callable
from itertools import chain
import re
from typing import List, Dict, Any

New dependencies:
- `srt` for handling subtitle files, can be installed via `pip install srt`
- `pylrc` for handling lrc files, can be installed via `pip install pylrc`

In [69]:
# Garbage collection
def release_cuda_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [70]:
# This is an example test function to save word by word subtitle files for visualization in SubtitleEdit, likely not needed for production
def output_subtitle(segments, output_file):
    subtitle_objects = []
    global_word_index = 1
    
    for segment in segments:
        words = segment.get("words", [])
        for word in words:
            start_time = word["start"]
            end_time = word["end"]
            text = word["word"]
            
            subtitle_objects.append(srt.Subtitle(index=global_word_index, start=srt.timedelta(seconds=start_time), end=srt.timedelta(seconds=end_time), content=text))
            global_word_index += 1

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(srt.compose(subtitle_objects))

In [151]:
# Useful functions to preprocess each lines of a lyrics or subtitle file
def preprocess_lrc_line(line: str) -> str | None:
    if ":" in line or line == "": return None # remove metadata lines like [00:01.00] or [ar:Artist Name] or empty lines
    return line.split('/')[0].strip() # remove Chinese translation if it exists, e.g. "Hello / 你好" -> "Hello"

In [ ]:
# Parsing files into segments

@dataclass
class Segment:
    text: str | None
    start: float
    end: float

def parse_srt(text, preprocess: Callable[[str], str | None]) -> list[Segment]:
    """Parse SRT subtitle text into a list of Segment objects, applying the provided preprocess function to the text content of each subtitle entry."""
    import srt
    subs = list(srt.parse(text))
    segments = [
        Segment(text=preprocess(s.content.replace("\n", " ")), start=s.start.total_seconds(), end=s.end.total_seconds())
        for s in subs
    ]
    return segments

def parse_lrc(text, preprocess: Callable[[str], str | None]) -> list[Segment]:
    """Parse LRC lyrics text into a list of Segment objects, applying the provided preprocess function to the text content of each lyric entry."""
    import pylrc
    lrc_subs = pylrc.parse(text)
    if not lrc_subs:
        return [Segment(text, start=0.0, end=0.0), False] # type: ignore # handle unsynced lyrics with no timestamps, treat the whole lyrics as one segment
    # the False is simply an indicator to indicate to the next step that the lyrics is unsynced
    segments = []
    for idx, lrc in enumerate(lrc_subs):
        start = lrc.time
        content = preprocess(lrc.text)
        if idx < len(lrc_subs) - 1:
            end = lrc_subs[idx + 1].time
        else:
            end = start + 5.0  # Assuming a fixed duration of 5 seconds for the last line
        segments.append(Segment(text=content, start=start, end=end))
    return segments

def flatten(segments: list[Segment], duration: float) -> list[Segment]:
    """Flatten a list of Segment objects into a single Segment that spans the entire duration, concatenating the text content of all segments. This would convert synced lyrics into unsynced lyrics that spans the duration. Useful since some synced lyrics file may not be in sync with music videos."""
    if not segments: return []
    return [Segment(text=" ".join([s.text for s in segments if s.text]), start=0.0, end=duration)]

In [225]:
# rebuild the segments with the new line-level alignment

from collections.abc import Iterable

def join_display_tokens(tokens: Iterable[str]) -> str:
    tokens = list(tokens)
    if any('\u4e00' <= ch <= '\u9fff' for token in tokens for ch in token):
        return ''.join(tokens)
    return ' '.join(tokens)

def make_segment(
        start: int, # start index
        end: int,  # end index
        words: list[dict[str, Any]] # list of word-level alignment results, from whisper output 
    ) -> dict[str, Any]:
    """
    [{start,end,text},...] -> [{start,end,text,words:[{word,start,end},...]}]
    """
    curr_words = [w for w in words[start:end] if w["word"].strip()]
    words_start = words[start]["start"] if words else 0.0
    words_end = words[end-1]["end"] if words else 0.0
    return {
        "start": words_start,
        "end": words_end,
        "text": join_display_tokens(w["word"] for w in curr_words),
        "words": curr_words
    }

In [222]:
TOKEN_RE = re.compile(r'[\u4e00-\u9fff]|[A-Za-z0-9]+|[^\s]')

def lyric_tokens(text: str) -> list[str]:
    return TOKEN_RE.findall(text)

def realign_easy(segments: list[Segment], words: list[dict]) -> list[dict[str, Any]]:
    """
    The length of words and segments match, so we can just assign words to each segment based on the word count of each segment
    """
    print(f"Segments: {[s.text for s in segments]}")
    print(f"Words: {[w['word'] for w in words]}")
    new_segments = []
    curr_idx = 0
    for segment in segments:
        segment_word_count = len(lyric_tokens(segment.text)) # type: ignore
        prev_idx = curr_idx
        curr_idx += segment_word_count
        print(f"Line: {segment.text}, Start: {prev_idx}, End: {curr_idx}")
        print(f"Aligned words for this line: {[w['word'] for w in words[prev_idx:curr_idx]]}")
        new_segments.append(make_segment(prev_idx, curr_idx, words))
    return new_segments 

def realign_hard(segments: list[Segment], words: list[dict]) -> list[dict[str, Any]]:
    """
    If the length of words in the original segments don't match the length of words in the whisper output
    """
    def clean_token(s: str) -> str:
        return re.sub(r"[^\w']+", "", s).lower()

    idxs = []
    new_segments = []
    w_idx = 0
    total_words = len(words)

    for lrc_line in segments:
        tokens = [t for t in (clean_token(t) for t in lyric_tokens(lrc_line.text)) if t]
        line_start = w_idx
        for tok in tokens:
            # advance until we find a matching cleaned token or run out of words
            while w_idx < total_words and clean_token(words[w_idx]["word"]) != tok:
                w_idx += 1
            if w_idx >= total_words:
                break
            # matched this token, consume the word
            w_idx += 1
        idxs.append((line_start, w_idx))
    for (start, end) in idxs:
        print(f"Line: {segments[idxs.index((start,end))].text}, Start: {start}, End: {end}")
        print(f"Aligned words for this line: {[w['word'] for w in words[start:end]]}")
        new_segments.append(make_segment(start, end, words))
    return new_segments

In [213]:
# This example uses simple dict lookup to cache loaded models, in FastAPI web app, different mechanism can be used
models = {}
transcription_models = {}

def align_and_output(
    input_audio_file: str|Path, # path to the input audio file, e.g. demucs separated vocal tracks
    input_subtitle_file: str|Path, # input .srt or .lrc file with or without timestamps 
    language_code: str | None = None, # optional language code for alignment, if not provided, it will be detected automatically using Whisper
    use_synced_subs: bool = False, # whether to use synced subtitles (True) or flatten them into unsynced lyrics (False), unsynced lyrics may give better results since for music videos, the subtitles may not be perfectly in sync with the vocals
    transcription_model_size: str = "tiny", # size of the Whisper model to use for language detection if language_code is not provided
    device: str = "cuda" # device to run the models on, e.g. "cuda" or "cpu"
):
    input_audio_file = Path(input_audio_file)
    input_subtitle_file = Path(input_subtitle_file)
    audio = whisperx.load_audio(str(input_audio_file))

    if not language_code: # use autodetection if language code is not provided
        before_langdetect_load = time.perf_counter()
        whisper_model = transcription_models.get(transcription_model_size)
        if not whisper_model: # load model and add to cache if not already loaded
            whisper_model = whisperx.load_model(transcription_model_size, device=device)
            transcription_models[transcription_model_size] = whisper_model
        after_langdetect_load = time.perf_counter()
        print(f"Whisper model load time for language detection: {after_langdetect_load - before_langdetect_load:.2f} seconds")
        before_langdetect = time.perf_counter()
        language_code = whisper_model.detect_language(audio)
        after_langdetect = time.perf_counter()
        print(f"Language detection time: {after_langdetect - before_langdetect:.2f} seconds")
        print(f"Detected language: {language_code}")

    before_load = time.perf_counter()
    if language_code not in models: # load align model and add to cache
        model_a, metadata = whisperx.load_align_model(language_code=language_code, device=device)
        models[language_code] = (model_a, metadata)
    else:
        model_a, metadata = models[language_code]
    after_load = time.perf_counter()
    print(f"Model load time: {after_load - before_load:.2f} seconds")

    with open(input_subtitle_file, "r", encoding="utf-8") as f:
        if input_subtitle_file.suffix == ".srt":
            sub_segments = parse_srt(f.read(), preprocess_lrc_line)
        elif input_subtitle_file.suffix == ".lrc":
            sub_segments = parse_lrc(f.read(), preprocess_lrc_line)
        else:
            raise ValueError("Unsupported subtitle format. Only .srt and .lrc are supported.")

    is_synced = True
    if sub_segments[-1] == False:
        is_synced = False
        sub_segments.pop() # remove the False indicator from the end of the list

    if not use_synced_subs:
        audio_length = audio.shape[0] / 16000
        subs = flatten(sub_segments, audio_length)
    else:
        subs = sub_segments
    segments = [s.__dict__ for s in subs] # convert list of Segment dataclass objects to list of dicts for whisperx input

    # sub_segments: list of Segment objects (aligned)
    # subs: list of Segment objects (can be flattened)

    before_align = time.perf_counter()
    result = whisperx.align(segments, model_a, metadata, audio, device=device,return_char_alignments=False, progress_callback=lambda progress: print(f"Alignment progress: {progress:.2f}%"))
    after_align = time.perf_counter()

    # original segments
    output_json_file = input_audio_file.with_suffix(".original_segments.json")
    with open(output_json_file, "w", encoding="utf-8") as f:
        json.dump(result['segments'], f, indent=4, ensure_ascii=False)

    # re-align the result so the lines matches the original subtitle lines
    before_realign = time.perf_counter()
    if is_synced: # resync line level alignment with original synced lyrics, since sometimes WhisperX output can output a line which consists of many lines
        word_segments = result['segments']
        words = [word for segment in word_segments for word in segment["words"]]
        # extra processing against synced lyrics
        sub_segments = [l for l in sub_segments if l.text and l.text.strip()]
        from itertools import chain
        lrcs_tokens = list(chain.from_iterable([l.text.split(" ") for l in sub_segments])) # type: ignore
        lrcs_tokens = [w["word"] for w in words]
        if len(lrcs_tokens) == len(words):
            # save sub_segments and words for debugging
            with open(input_audio_file.with_suffix(".sub_segments.json"), "w", encoding="utf-8") as f:
                json.dump([s.__dict__ for s in sub_segments], f, indent=4, ensure_ascii=False)
            with open(input_audio_file.with_suffix(".words.json"), "w", encoding="utf-8") as f:
                json.dump(words, f, indent=4, ensure_ascii=False)
            segments = realign_easy(sub_segments, words)
        else:
            print("Length mismatch between aligned words and original subtitle tokens, using hard realignment")
            segments = realign_hard(sub_segments, words)
    else:
        segments = result['segments']
    after_realign = time.perf_counter()
    print(f"Realignment time: {after_realign - before_realign:.2f} seconds")

    # saving the json and srt file is for testing only, not needed for production
    output_json_file = input_audio_file.with_suffix(".aligned.json")
    output_subtitle_file = input_audio_file.with_suffix(".words.srt")
    output_subtitle(segments, output_subtitle_file)
    with open(output_json_file, "w", encoding="utf-8") as f:
        json.dump(segments, f, indent=4, ensure_ascii=False)
    print(f"Alignment time: {after_align - before_align:.2f} seconds")
    print(f"Total time: {after_realign - before_align:.2f} seconds")

    return segments


In [ ]:
align_and_output("examples/hk_kissingeverywhere.wav", "examples/hk_kissingeverywhere.lrc", language_code=None, use_synced_subs=False, device="cuda")

Here is an example JSON output of the aligned segments (after re-aligning with lyric lines)
```json
[
    {
        "start": 39.222,
        "end": 50.403,
        "text": "I'm at a payphone trying to call home All of my change I spent on you Where have the times gone?",
        "words": [
            {
                "word": "I'm",
                "start": 39.222,
                "end": 39.782,
                "score": 0.503
            },
            {
                "word": "at",
                "start": 39.942,
                "end": 40.082,
                "score": 0.624
            },
            {
                "word": "a",
                "start": 40.182,
                "end": 40.263,
                "score": 0.621
            },
```